In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import json
import cv2
import numpy as np
from collections import defaultdict

IMAGE_DIR = "/content/drive/MyDrive/Camscanner/Camscanner_Dataset"
GT_JSON   = "/content/drive/MyDrive/Camscanner/Ground_Truth/ground_truth.json"

with open(GT_JSON) as f:
    ground_truth = json.load(f)

image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(".png")])

issues = defaultdict(list)
passed = []

print(f"Total images in folder   : {len(image_files)}")
print(f"Total annotated entries  : {len(ground_truth)}")
print("-" * 50)

for fname in image_files:
    path = os.path.join(IMAGE_DIR, fname)

    # ── Check 1: Image readable ──────────────────────────
    img = cv2.imread(path)
    if img is None:
        issues["corrupt"].append(fname)
        continue

    H, W = img.shape[:2]

    # ── Check 2: Annotation exists ───────────────────────
    if fname not in ground_truth:
        issues["missing_annotation"].append(fname)
        continue

    pts = ground_truth[fname]

    # ── Check 3: Exactly 4 points ────────────────────────
    if len(pts) != 4:
        issues["wrong_point_count"].append((fname, len(pts)))
        continue

    pts = np.array(pts)

    # ── Check 4: No duplicate points ─────────────────────
    if len(np.unique(pts, axis=0)) < 4:
        issues["duplicate_points"].append(fname)

    # ── Check 5: Points inside image bounds ──────────────
    if np.any(pts[:, 0] < 0) or np.any(pts[:, 0] >= W) or \
       np.any(pts[:, 1] < 0) or np.any(pts[:, 1] >= H):
        issues["out_of_bounds"].append(fname)

    # ── Check 6: Quad area large enough ──────────────────
    area = cv2.contourArea(pts.reshape(-1, 1, 2).astype(np.float32))
    img_area = H * W
    if area < 0.05 * img_area:
        issues["quad_too_small"].append((fname, f"{100*area/img_area:.1f}%"))

    # ── Check 7: Quad is convex ───────────────────────────
    if not cv2.isContourConvex(pts.reshape(-1, 1, 2).astype(np.float32)):
        issues["non_convex_quad"].append(fname)

    # ── Check 8: Image resolution reasonable ─────────────
    if H < 256 or W < 256:
        issues["low_resolution"].append((fname, f"{W}x{H}"))

    # ── Check 9: Image not blank / mostly black ───────────
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if gray.mean() < 10:
        issues["blank_image"].append(fname)

    if not any(fname in str(v) for v in issues.values()):
        passed.append(fname)


# ── Report ────────────────────────────────────────────────
print(f"\n✅ Passed all checks : {len(passed)}")
print(f"❌ Has issues        : {sum(len(v) for v in issues.values())}\n")

labels = {
    "corrupt":            "Corrupt / unreadable images",
    "missing_annotation": "Images with no annotation",
    "wrong_point_count":  "Annotations without exactly 4 points",
    "duplicate_points":   "Annotations with duplicate corners",
    "out_of_bounds":      "Corner points outside image bounds",
    "quad_too_small":     "Quad covers less than 5% of image",
    "non_convex_quad":    "Non-convex quadrilaterals",
    "low_resolution":     "Images smaller than 256×256",
    "blank_image":        "Blank or nearly black images",
}

for key, label in labels.items():
    items = issues[key]
    if items:
        print(f"  [{key}] {label}: {len(items)}")
        for item in items[:3]:
            print(f"    → {item}")
        if len(items) > 3:
            print(f"    ... and {len(items)-3} more")

# ── Also catch annotated entries with no image file ───────
annotated_no_image = [f for f in ground_truth if f not in set(image_files)]
if annotated_no_image:
    print(f"\n  [orphan_annotations] Annotated but image missing: {len(annotated_no_image)}")
    for f in annotated_no_image[:3]:
        print(f"    → {f}")

Total images in folder   : 33
Total annotated entries  : 33
--------------------------------------------------

✅ Passed all checks : 33
❌ Has issues        : 0

